<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_2_2%D8%8C_%D8%B7%D8%B1%D9%8A%D9%82%D8%A9_CUPED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 2.2: طريقة CUPED

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الثاني

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 2.2».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import statsmodels.formula.api as smf
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest, confint_proportions_2indep
from statsmodels.stats.multitest import multipletests


def tidy(fit, terms=None):
    """جدول مختصر لنتيجة الانحدار: المعامل، والخطأ المعياري، وفترة الثقة، وقيمة p."""
    ci = fit.conf_int()
    out = pd.DataFrame({"المعامل": fit.params, "SE": fit.bse, "أدنى 95%": ci[0], "أعلى 95%": ci[1], "p": fit.pvalues})
    return out.loc[terms] if terms else out


say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [ ]:
def simulate_cuped(n=20_000, effect=2.0, rho=0.7, seed=4):
    r = np.random.default_rng(seed)
    treatment = r.binomial(1, 0.5, n)
    pre = r.normal(100, 30, n)                                          # الإنفاق قبل التجربة
    noise = r.normal(0, 30 * np.sqrt(1 - rho ** 2), n)
    post = 100 + rho * (pre - 100) + noise + effect * treatment         # الإنفاق أثناء التجربة
    return pd.DataFrame({"treatment": treatment, "pre": pre, "post": post})


cu = simulate_cuped()
theta = np.cov(cu["post"], cu["pre"])[0, 1] / cu["pre"].var(ddof=1)
cu["post_cuped"] = cu["post"] - theta * (cu["pre"] - cu["pre"].mean())

rows = []
for col in ["post", "post_cuped"]:
    fit = smf.ols(f"{col} ~ treatment", data=cu).fit(cov_type="HC1")
    lo, hi = fit.conf_int().loc["treatment"]
    rows.append({"المقياس": col, "الأثر": fit.params["treatment"], "SE": fit.bse["treatment"],
                 "أدنى 95%": lo, "أعلى 95%": hi})
res = pd.DataFrame(rows).set_index("المقياس")
reduction = 1 - (res.loc["post_cuped", "SE"] / res.loc["post", "SE"]) ** 2
say(f"قيمة θ = {theta:.3f}، وانخفض التباين بنسبة {reduction:.0%} تقريبًا، والمتوقع ρ² = {0.7 ** 2:.0%}")
res

,الأثر,SE,أدنى 95%,أعلى 95%
المقياس,,,,
post,1.7716,0.4223,0.9438,2.5993
post_cuped,1.7143,0.3022,1.1220,2.3066


<div dir="rtl">

## المطلوب

1. قدّر الأثر بالانحدار `post ~ treatment + pre` وقارن خطأه المعياري بخطأ CUPED. ماذا تلاحظ؟
2. تجربة تحتاج 40,000 مستخدم بدون CUPED. كم مستخدمًا تحتاج معها إذا كان ρ يساوي 0.5 ثم 0.7 ثم 0.9؟

</div>

In [ ]:
# 1)
fit = smf.ols(f"{col} ~ treatment", data=cu).fit(cov_type="HC1")
print("الخطأ المعياري للانحدار:", fit.bse["treatment"].round(4)) # الخطا المعياري يساوي خطا cuped (0.3022)
# 2) العينة الجديدة تساوي تقريبًا العينة القديمة مضروبة في (1 - ρ²)
n = 40_000
p05 = n * (1 - (0.5 * 0.5))
p07 = n * (1 - (0.7 * 0.7))
p09 = n * (1 - (0.9 * 0.9))
print("p05 =", round(p05)) # قيمة العينة الجديده اذا كانت ρ 0.5
print("p07 =", round(p07)) # قيمة العينة الجديده اذا كانت ρ 0.7
print("p09 =", round(p09)) # قيمة العينة الجديده اذا كانت ρ 0.9


الخطأ المعياري للانحدار: 0.3022
p05 = 30000
p07 = 20400
p09 = 7600


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 2.2».

</div>